# Episode 4: Task & Sub-Agent Delegation

One agent can do a lot. But some work is better split: research it, then write it. In this episode, one **coordinator** agent delegates to **specialist** agents.

**In this episode you'll build:** a coordinator that uses a researcher and a writer as tools.

> **Not multi-agent (yet).** This is *task delegation* — one agent drives, calling others as tools. True multi-agent systems, where autonomous agents talk as peers over a Hub, start in Group 3. The distinction matters, so we keep the names separate.

## The idea: an agent can be a tool

In Episode 3 a tool was a plain function. It turns out **an agent can also be a tool**.

`agent.as_tool(description=...)` wraps an agent so another agent can call it. The coordinator's model sees it exactly like any other tool — it decides when to delegate, and to whom.

## Step 1: Create specialist agents

Two focused agents, each with a narrow job.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

researcher = Agent(
    "researcher",
    prompt="You are a research specialist. Give 3 concise bullet findings.",
    config=config,
)
writer = Agent(
    "writer",
    prompt="You are a writer. Turn findings into a short 3-sentence paragraph.",
    config=config,
)

## Step 2: Wire them into a coordinator

The coordinator gets the two specialists as tools via `as_tool()`. The `description` is what the coordinator's model reads to decide when to call each one — so write it like a tool docstring.

In [ ]:
coordinator = Agent(
    "coordinator",
    prompt="Use researcher to gather findings, then writer to produce the paragraph.",
    config=config,
    tools=[
        researcher.as_tool(description="Research a topic and return findings."),
        writer.as_tool(description="Write a paragraph from research findings."),
    ],
)

## Step 3: Run it

We ask the coordinator once. Behind the scenes it calls `researcher`, passes the findings to `writer`, and returns the finished paragraph.

In [1]:
reply = await coordinator.ask("Create a short paragraph on benefits of multi-agent AI.")
print(reply.body)

Multi-agent AI systems offer significant benefits by enabling collaboration and task division, which enhances problem-solving capabilities. They improve scalability and flexibility by allowing multiple agents to operate concurrently in dynamic environments. Additionally, these systems promote robustness and fault tolerance, as they can continue functioning effectively even if individual agents fail.


## TaskConfig: auto-delegation

Wiring agents by hand is explicit and clear. For general delegation there's a shortcut: pass `tasks=TaskConfig()` and the agent gets two tools **for free** — `run_subtask` (one task) and `run_subtasks` (many, optionally in parallel).

The agent spawns its own sub-agents per task; you don't pre-build them.

In [2]:
from autogen.beta import TaskConfig

planner = Agent(
    "planner",
    prompt="Break the request into subtasks and run them with your run_subtasks tool.",
    config=config,
    tasks=TaskConfig(),
)

reply = await planner.ask("Name two primary colors. Use one subtask per color.")
print(reply.body)

Two primary colors are red and blue.


## Delegation vs networking

Keep these two mental models separate — the rest of the workshop depends on it:

| | Task delegation (this episode) | Multi-agent networking (Group 3) |
|---|---|---|
| Who's in charge | one coordinator drives | autonomous peers |
| How agents connect | `as_tool()` / `TaskConfig` — function calls | Hub channels |
| Best for | "do these steps for me" | open collaboration, debate, routing |

Use `as_tool()` when one agent should own the workflow. Reach for the Hub when agents are genuine peers.

## What's next

You've met the `Agent` class and made agents work together as tools. Next is something different — the **`LiveAgent`**, a real-time voice agent that works nothing like `Agent`.